# Modelos de clasificación para aprendizaje automático relacional

En este notebook se entrenan y comparan varios modelos de clasificación para predecir la clase de los nodos del dataset Cora.

Se comparan tres conjuntos de características:

- **Features relacionales**: métricas calculadas a partir de la estructura del grafo.
- **Features nativas**: atributos propios de los nodos, representados como variables `word_*`.
- **Features combinadas**: unión de las features relacionales y las nativas.

Los modelos utilizados son:

- Árbol de decisión.
- KNN.
- Random Forest.

Además, se aplica ajuste de hiperparámetros mediante `GridSearchCV` sobre algunos modelos.

In [ ]:
# %pip install pandas numpy scikit-learn matplotlib joblib

# ============================================
# 1. IMPORTS
# ============================================

import os
import joblib

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix


## 1. Carga del dataset

Se carga el fichero `features.csv`, que contiene una fila por cada nodo del grafo.  
Cada fila incluye métricas relacionales, atributos nativos del nodo y la clase que se quiere predecir.

In [ ]:
# ============================================
# 2. CARGA DEL DATASET
# ============================================

df = pd.read_csv("../data/features.csv")

print("Tamaño del dataset:", df.shape)
df.head()

## 2. Exploración básica

Antes de entrenar los modelos, se revisan las columnas disponibles, la distribución de clases y la existencia de valores nulos.

In [ ]:
# ============================================
# 3. EXPLORACIÓN BÁSICA
# ============================================

print("Primeras columnas:")
print(df.columns[:15].tolist())

print("Número total de columnas:", len(df.columns))

print("Distribución de clases:")
print(df["class"].value_counts())

print("Valores nulos totales:", df.isnull().sum().sum())

## 3. Separación de características

Se separan las variables predictoras en tres grupos:

- **Relacionales**: métricas obtenidas a partir del grafo.
- **Nativas**: columnas `word_*`, que representan características propias de los artículos.
- **Combinadas**: unión de las anteriores.

La variable objetivo es `class`.

In [ ]:
# ============================================
# 4. SEPARACIÓN DE FEATURES
# ============================================

target = "class"

features_relacionales = [
    "degree",
    "closeness",
    "betweenness",
    "pagerank",
    "eigenvector_centrality",
    "clustering_coef",
    "triangles",
    "kcore",
    "community_louvain"
]

features_nativas = [col for col in df.columns if col.startswith("word_")]

features_combinadas = features_relacionales + features_nativas

X_rel = df[features_relacionales]
X_nat = df[features_nativas]
X_comb = df[features_combinadas]

y = df[target]

print("Features relacionales:", X_rel.shape)
print("Features nativas:", X_nat.shape)
print("Features combinadas:", X_comb.shape)
print("Variable objetivo:", y.shape)

## 4. Función de entrenamiento y evaluación

Se define una función auxiliar para entrenar un modelo, evaluarlo sobre un conjunto de prueba y guardar sus resultados.

Las métricas utilizadas son:

- Accuracy.
- Precision macro.
- Recall macro.
- F1 macro.
- Matriz de confusión.

Se utiliza F1 macro porque el problema es multiclase y esta métrica da el mismo peso a todas las clases.


In [ ]:
# ============================================
# 5. FUNCIÓN DE ENTRENAMIENTO Y EVALUACIÓN
# ============================================

def entrenar_y_evaluar(nombre_modelo, modelo, nombre_features, X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )
    
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average="macro", zero_division=0)
    recall = recall_score(y_test, y_pred, average="macro", zero_division=0)
    f1_macro = f1_score(y_test, y_pred, average="macro", zero_division=0)
    matriz = confusion_matrix(y_test, y_pred)
    
    print("=" * 70)
    print("Modelo:", nombre_modelo)
    print("Features:", nombre_features)
    print("=" * 70)
    print("Accuracy:", accuracy)
    print("Precision macro:", precision)
    print("Recall macro:", recall)
    print("F1 macro:", f1_macro)
    print("\nMatriz de confusión:")
    print(matriz)
    
    return {
        "modelo": nombre_modelo,
        "features": nombre_features,
        "accuracy": accuracy,
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1_macro
    }

## 5. Árbol de decisión

Primero se entrena un árbol de decisión con los tres conjuntos de características para comprobar qué tipo de información resulta más útil.

In [ ]:
# ============================================
# 6. MODELO BASE: ÁRBOL DE DECISIÓN
# ============================================

resultados = []

resultados.append(entrenar_y_evaluar(
    "Decision Tree",
    DecisionTreeClassifier(random_state=42),
    "relacionales",
    X_rel,
    y
))

resultados.append(entrenar_y_evaluar(
    "Decision Tree",
    DecisionTreeClassifier(random_state=42),
    "nativas",
    X_nat,
    y
))

resultados.append(entrenar_y_evaluar(
    "Decision Tree",
    DecisionTreeClassifier(random_state=42),
    "combinadas",
    X_comb,
    y
))

## 6. KNN

A continuación se entrena un clasificador KNN.  
Como KNN se basa en distancias, se utiliza `StandardScaler` dentro de un `Pipeline` para escalar las variables antes del entrenamiento.

In [ ]:
# ============================================
# 7. MODELO BASE: KNN
# ============================================

resultados.append(entrenar_y_evaluar(
    "KNN",
    Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier())
    ]),
    "relacionales",
    X_rel,
    y
))

resultados.append(entrenar_y_evaluar(
    "KNN",
    Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier())
    ]),
    "nativas",
    X_nat,
    y
))

resultados.append(entrenar_y_evaluar(
    "KNN",
    Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier())
    ]),
    "combinadas",
    X_comb,
    y
))

## 7. Ajuste de hiperparámetros del árbol de decisión

Se aplica `GridSearchCV` sobre el árbol de decisión utilizando las features combinadas, ya que fueron las que dieron mejor resultado inicial.

La métrica utilizada para seleccionar los hiperparámetros es `f1_macro`.


In [ ]:
# ============================================
# 8. AJUSTE DE HIPERPARÁMETROS: DECISION TREE
# ============================================

param_grid_arbol = {
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

grid_arbol = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid_arbol,
    cv=5,
    scoring="f1_macro"
)

grid_arbol.fit(X_comb, y)

print("Mejores hiperparámetros:")
print(grid_arbol.best_params_)

print("Mejor F1 macro medio en validación cruzada:")
print(grid_arbol.best_score_)


In [ ]:
# ============================================
# 9. EVALUACIÓN DEL ÁRBOL OPTIMIZADO
# ============================================

mejor_arbol = DecisionTreeClassifier(
    max_depth=grid_arbol.best_params_["max_depth"],
    min_samples_leaf=grid_arbol.best_params_["min_samples_leaf"],
    min_samples_split=grid_arbol.best_params_["min_samples_split"],
    random_state=42
)

resultados.append(entrenar_y_evaluar(
    "Decision Tree GridSearch",
    mejor_arbol,
    "combinadas",
    X_comb,
    y
))

## 8. Ajuste de hiperparámetros de KNN

Se aplica `GridSearchCV` sobre KNN usando las features relacionales, ya que fueron las que mejor funcionaron para este modelo en la evaluación inicial.

La métrica utilizada para seleccionar los hiperparámetros es `f1_macro`.


In [ ]:
# ============================================
# 10. AJUSTE DE HIPERPARÁMETROS: KNN
# ============================================

pipeline_knn = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier())
])

param_grid_knn = {
    "knn__n_neighbors": [3, 5, 7, 9, 11],
    "knn__weights": ["uniform", "distance"],
    "knn__metric": ["euclidean", "manhattan"]
}

grid_knn = GridSearchCV(
    pipeline_knn,
    param_grid_knn,
    cv=5,
    scoring="f1_macro"
)

grid_knn.fit(X_rel, y)

print("Mejores hiperparámetros:")
print(grid_knn.best_params_)

print("Mejor F1 macro medio en validación cruzada:")
print(grid_knn.best_score_)


In [ ]:
# ============================================
# 11. EVALUACIÓN DEL KNN OPTIMIZADO
# ============================================

mejor_knn = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(
        n_neighbors=grid_knn.best_params_["knn__n_neighbors"],
        weights=grid_knn.best_params_["knn__weights"],
        metric=grid_knn.best_params_["knn__metric"]
    ))
])

resultados.append(entrenar_y_evaluar(
    "KNN GridSearch",
    mejor_knn,
    "relacionales",
    X_rel,
    y
))

## 9. Random Forest

Se entrena un Random Forest como modelo adicional.  
Este modelo combina varios árboles de decisión, lo que suele mejorar la capacidad de generalización frente a un único árbol.

In [ ]:
# ============================================
# 12. MODELO BASE: RANDOM FOREST
# ============================================

resultados.append(entrenar_y_evaluar(
    "Random Forest",
    RandomForestClassifier(random_state=42),
    "relacionales",
    X_rel,
    y
))

resultados.append(entrenar_y_evaluar(
    "Random Forest",
    RandomForestClassifier(random_state=42),
    "nativas",
    X_nat,
    y
))

resultados.append(entrenar_y_evaluar(
    "Random Forest",
    RandomForestClassifier(random_state=42),
    "combinadas",
    X_comb,
    y
))

## 10. Ajuste de hiperparámetros de Random Forest

Como Random Forest con features combinadas obtuvo los mejores resultados iniciales, se aplica `GridSearchCV` para ajustar sus hiperparámetros.

La métrica utilizada para seleccionar los hiperparámetros es `f1_macro`.


In [ ]:
# ============================================
# 13. AJUSTE DE HIPERPARÁMETROS: RANDOM FOREST
# ============================================

param_grid_rf = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid_rf,
    cv=5,
    scoring="f1_macro"
)

grid_rf.fit(X_comb, y)

print("Mejores hiperparámetros:")
print(grid_rf.best_params_)

print("Mejor F1 macro medio en validación cruzada:")
print(grid_rf.best_score_)


In [ ]:
# ============================================
# 14. EVALUACIÓN DEL RANDOM FOREST OPTIMIZADO
# ============================================

mejor_rf = RandomForestClassifier(
    max_depth=grid_rf.best_params_["max_depth"],
    min_samples_leaf=grid_rf.best_params_["min_samples_leaf"],
    min_samples_split=grid_rf.best_params_["min_samples_split"],
    n_estimators=grid_rf.best_params_["n_estimators"],
    random_state=42
)

resultados.append(entrenar_y_evaluar(
    "Random Forest GridSearch",
    mejor_rf,
    "combinadas",
    X_comb,
    y
))

## 11. Comparación final de resultados

Se construye una tabla final con todos los modelos evaluados, ordenada por accuracy.

In [ ]:
# ============================================
# 15. TABLA FINAL DE RESULTADOS
# ============================================

tabla_resultados = pd.DataFrame(resultados)

tabla_resultados = tabla_resultados.sort_values(
    by="f1_macro",
    ascending=False
)

tabla_resultados


## 12. Serialización del modelo final

El modelo final seleccionado es Random Forest optimizado con `GridSearchCV` y entrenado con features combinadas.

Para poder reutilizarlo en un entorno limpio, se vuelve a entrenar con todos los datos disponibles y se guarda con `joblib.dump`.


In [ ]:
# ============================================
# 16. SERIALIZACIÓN DEL MODELO FINAL
# ============================================

modelo_final = RandomForestClassifier(
    max_depth=grid_rf.best_params_["max_depth"],
    min_samples_leaf=grid_rf.best_params_["min_samples_leaf"],
    min_samples_split=grid_rf.best_params_["min_samples_split"],
    n_estimators=grid_rf.best_params_["n_estimators"],
    random_state=42
)

modelo_final.fit(X_comb, y)

os.makedirs("../modelo_final", exist_ok=True)

ruta_modelo = "../modelo_final/modelo_final_random_forest.joblib"

joblib.dump({
    "modelo": modelo_final,
    "features": features_combinadas,
    "target": target
}, ruta_modelo)

print("Modelo final guardado en:", ruta_modelo)


## 13. Comprobación de carga del modelo final

Se comprueba que el fichero guardado puede cargarse correctamente y que el modelo es capaz de realizar predicciones.


In [ ]:
# ============================================
# 17. COMPROBACIÓN DE CARGA DEL MODELO FINAL
# ============================================

modelo_guardado = joblib.load("../modelo_final/modelo_final_random_forest.joblib")

modelo_cargado = modelo_guardado["modelo"]
features_modelo = modelo_guardado["features"]

X_prueba = df[features_modelo].head()

predicciones = modelo_cargado.predict(X_prueba)

print("Modelo cargado correctamente.")
print("Predicciones de prueba:")
print(predicciones)


## 14. Conclusiones preliminares

El mejor modelo obtenido es Random Forest con ajuste de hiperparámetros mediante `GridSearchCV` y usando features combinadas.

Los resultados muestran que combinar las métricas relacionales del grafo con las características nativas de los nodos proporciona mejores resultados que usar cada grupo de características por separado.

KNN obtiene resultados claramente inferiores al árbol de decisión y a Random Forest, incluso tras el ajuste de hiperparámetros. Esto puede deberse a que KNN depende de distancias entre ejemplos y el problema tiene un número elevado de dimensiones, especialmente al usar las features nativas o combinadas.

El modelo final se guarda en la carpeta `modelo_final` mediante `joblib.dump`, y se incluye una comprobación de carga para asegurar que puede reutilizarse correctamente.
